## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

import joblib

## 2. Load Dataset

In [2]:
df = pd.read_csv("../dataset/cardio_train.csv", sep=';')

print(df.shape)
print(df.isnull().sum())
print(df.duplicated().sum())

(70000, 13)
id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64
0


## 3. Basic Cleaning

In [3]:
df = df.drop_duplicates()

In [4]:
df["age_years"] = (df["age"] / 365.25).astype(int)
df.drop(columns=["age"], inplace=True)

## 4. Medical Outlier Filtering

In [5]:
df = df[
    (df["ap_hi"].between(70, 250)) &
    (df["ap_lo"].between(40, 200)) &
    (df["height"].between(120, 220)) &
    (df["weight"].between(35, 200))
]

## 5. Lock Feature Order

In [6]:
FEATURES = [
    "age_years",
    "gender",
    "height",
    "weight",
    "ap_hi",
    "ap_lo",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active"
]

TARGET = "cardio"

X = df[FEATURES]
y = df[TARGET]

## 6. Startified Train-Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## 7. Preprocessor

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), FEATURES)
    ]
)

## 8. Define Models To Compare

In [9]:
models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42
    ),
    "GradientBoost": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
}

## 9. Train + Evaluate All Models

In [10]:
results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    probs = pipeline.predict_proba(X_test)[:, 1]
    preds = pipeline.predict(X_test)
    
    std_prob = probs.std()
    
    print(f"\n{name}")
    print("Probability STD:", std_prob)
    
    if std_prob < 0.02:
        print("❌ INVALID MODEL (flat probabilities)")
        continue
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "ROC_AUC": roc_auc_score(y_test, probs)
    })


Logistic
Probability STD: 0.24203968969350123

RandomForest
Probability STD: 0.2511018848774343

GradientBoost
Probability STD: 0.26008764972856946

XGBoost
Probability STD: 0.26360902


In [11]:
results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
results_df

,Model,Accuracy,ROC_AUC
3,XGBoost,0.737156,0.803794
2,GradientBoost,0.737593,0.802907
1,RandomForest,0.733882,0.802360
0,Logistic,0.727623,0.791337


## 10. Select Best Model

In [12]:
best_model_name = results_df.iloc[0]["Model"]
best_base_model = models[best_model_name]

## 11. Hyperparameter Tunning

In [14]:
xgb_safe = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",   # 🔑 CRITICAL
    n_jobs=1,             # 🔑 CRITICAL
    eval_metric="logloss",
    random_state=42
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", xgb_safe)
])

param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [3, 4],
    "model__learning_rate": [0.05, 0.1]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=1,              # 🔑 CRITICAL
    error_score="raise"    # 🔑 Forces real error if any
)

grid.fit(X_train, y_train)

best_pipeline = grid.best_estimator_

## 12. Probability Calibration

In [15]:
calibrated_model = CalibratedClassifierCV(
    best_pipeline,
    method="sigmoid",
    cv=3
)

calibrated_model.fit(X_train, y_train)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2","Pipeline(step...=None, ...))])"
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the col

## 13. Final Validation

In [16]:
final_probs = calibrated_model.predict_proba(X_test)[:, 1]
final_preds = calibrated_model.predict(X_test)

print("Final Probability STD:", final_probs.std())
print("ROC-AUC:", roc_auc_score(y_test, final_probs))
print(confusion_matrix(y_test, final_preds))
print(classification_report(y_test, final_preds))

Final Probability STD: 0.26336784088114873
ROC-AUC: 0.8041349356214955
[[5479 1462]
 [2139 4662]]
              precision    recall  f1-score   support

           0       0.72      0.79      0.75      6941
           1       0.76      0.69      0.72      6801

    accuracy                           0.74     13742
   macro avg       0.74      0.74      0.74     13742
weighted avg       0.74      0.74      0.74     13742



## 14. Save Final Model

In [20]:
joblib.dump(calibrated_model, "cardio_final_safe_model.pkl", compress=3)

['cardio_final_safe_model.pkl']

## 15. Safe Prefiction

In [18]:
model = joblib.load("cardio_final_safe_model.pkl")

sample = pd.DataFrame([{
    "age_years": 68,
    "gender": 2,
    "height": 160,
    "weight": 105,
    "ap_hi": 200,
    "ap_lo": 130,
    "cholesterol": 3,
    "gluc": 3,
    "smoke": 1,
    "alco": 1,
    "active": 0
}])

assert list(sample.columns) == FEATURES

probability = model.predict_proba(sample)[0][1]
probability

np.float64(0.8523968191334071)